In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
from src.config import FIGURES_DIR
from src.data_utils import load_raw_data
from src.preprocessing import split_train_test, build_pipeline_a, build_pipeline_b

# 1. Đọc dữ liệu thô và tách Train/Test
df_raw = load_raw_data()
X_train, X_test, y_train, y_test = split_train_test(df_raw)

print(f"Kích thước tập Train: {X_train.shape} | Tỷ lệ Bad (1): {y_train.mean()*100:.1f}%")
print(f"Kích thước tập Test : {X_test.shape} | Tỷ lệ Bad (1): {y_test.mean()*100:.1f}%")

# 2. Fit trên tập Train và Transform trên cả Train/Test cho Luồng A
pipe_a = build_pipeline_a()
X_train_a = pipe_a.fit_transform(X_train, y_train)
X_test_a = pipe_a.transform(X_test)

# 3. Fit trên tập Train và Transform trên cả Train/Test cho Luồng B
pipe_b = build_pipeline_b()
X_train_b = pipe_b.fit_transform(X_train, y_train)
X_test_b = pipe_b.transform(X_test)

print(f"\nSố chiều đặc trưng sau xử lý của Luồng A (Cơ bản)  : {X_train_a.shape[1]} cột")
print(f"Số chiều đặc trưng sau xử lý của Luồng B (Cải tiến): {X_train_b.shape[1]} cột")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Biểu đồ 1: Dữ liệu gốc của credit_amount
sns.histplot(X_train["credit_amount"], kde=True, ax=axes[0], color="#3498db")
axes[0].set_title(f"Gốc: credit_amount (Skew = {X_train['credit_amount'].skew():.2f})", fontweight="bold")

# Biểu đồ 2: Sau StandardScaler của Luồng A (cột số thứ 2 trong NUMERICAL_COLS)
credit_amt_a = X_train_a[:, 1]
sns.histplot(credit_amt_a, kde=True, ax=axes[1], color="#f39c12")
axes[1].set_title(f"Luồng A (StandardScaler): Skew = {pd.Series(credit_amt_a).skew():.2f}", fontweight="bold")

# Biểu đồ 3: Sau PowerTransformer + RobustScaler của Luồng B
credit_amt_b = X_train_b[:, 1]
sns.histplot(credit_amt_b, kde=True, ax=axes[2], color="#2ecc71")
axes[2].set_title(f"Luồng B (Power + Robust): Skew = {pd.Series(credit_amt_b).skew():.2f}", fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "06_preprocessing_comparison.png", dpi=300)
plt.show()